In [43]:
import numpyro
import numpyro.distributions as dist
from numpyro.contrib.control_flow import scan
from numpyro.infer import MCMC, NUTS
from numpyro.infer import Predictive
# posterior samples
import numpy as np
from scipy.stats import norm, gaussian_kde
import arviz as az
import jax
import jax.numpy as jnp
from jax import random
import jax.random as random
from tqdm import tqdm
import pandas as pd
import numpy as np
import re
from datetime import datetime
from IPython.display import Markdown, display, HTML
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib as mpl
from matplotlib.lines import Line2D
import seaborn as sns
from statsmodels.tsa.filters.hp_filter import hpfilter

import warnings
warnings.simplefilter('ignore')
def get_model_number(model_str):
    nums = re.findall(r'\d+', model_str)
    if nums:
        return int(nums[-1])
    else:
        return None
colors = ['black', 'blue', 'orange', 'green', 'red', 'purple', 'brown', 'pink', 'gray']

from data_build import build_dataset
data = build_dataset("../data").dropna()

In [44]:
# for MCMC
pi = jnp.array(data['pi_cpi'],dtype=jnp.float64)
pi_prev = jnp.array(data['pi_cpi_prev'],dtype=jnp.float64)
pi_expect = jnp.array(data['Epi_spf_cpi'],dtype=jnp.float64)
N = jnp.array(data['N'],dtype=jnp.float64)
Nhat = jnp.array(data['N_BN_cycle'],dtype=jnp.float64)
Nbar = jnp.array(data['N_BN_trend'],dtype=jnp.float64)
Y = jnp.array(data['output'],dtype=jnp.float64)
x_markup = jnp.array(1/data['markup'],dtype=jnp.float64)
x_output_gap = jnp.array(data['output_gap_BN'],dtype=jnp.float64)
x_markup_bn = jnp.array(data['markup_BN_inv'],dtype=jnp.float64)
x_unempgap = jnp.array(data['unemp_gap'],dtype=jnp.float64)

## HSA, time-variant kappa, joint decomposition
$$
\pi_{t}	=\alpha\pi_{t-1}+(1-\alpha)\mathbb{E}_{t}\pi_{t+1}+\kappa_{t} x_{t}-\theta\hat{N}_{t}+v_{t}\\
x_{t}   = \phi_1 x_{t-1} + \phi_2 \hat{N_{t}}+e_{t}\\
N_{t}	=\hat{N_{t}}+\bar{N_{t}}\\
\hat{N_{t}}	=\rho_{1}\hat{N}_{t-1}+\rho_{2}\hat{N}_{t-2}+u_{t}\\
\bar{N_{t}}	=n+\bar{N}_{t-1}+\epsilon_{t}\\
\kappa_{t}	=\kappa_{t-1}+\delta \Delta \bar{N}_{t}
$$

# Prior distributions

In [45]:
alpha_mu = 0.5
alpha_sigma = 0.2
kappa_mu = 0.1
kappa_sigma = 0.2
theta_mu = 0.1
theta_sigma = 0.2
delta_mu = 0.1
delta_sigma = 0.2
beta_mu = 0.1
beta_sigma = 0.2
phi_mu = 0.1
phi_sigma = 0.2

def set_prior_distributions():
    priors = {
        # NKPC params
        "alpha"      : dist.Normal(alpha_mu, alpha_sigma),
        "kappa"      : dist.Normal(kappa_mu, kappa_sigma),
        "kappa0"     : dist.Normal(kappa_mu, kappa_sigma),
        "theta"      : dist.Normal(theta_mu, theta_sigma), 
        "delta"      : dist.Normal(delta_mu, delta_sigma),  
        "beta"       : dist.Normal(beta_mu, beta_sigma), 
        "phi"        : dist.Normal(phi_mu, phi_sigma),
        # Sigma
        "n"          : dist.Normal(0, 1), 
        # non informative priors
        "sigma_u"    : dist.InverseGamma(0.001, 0.001),  
        "sigma_eps"  : dist.InverseGamma(0.001, 0.001),  
        "sigma_v"    : dist.InverseGamma(0.001, 0.001),  
        "sigma_mu"   : dist.InverseGamma(0.001, 0.001),  
        "sigma_e"    : dist.InverseGamma(0.001, 0.001),  
        "sigma_eta"  : dist.InverseGamma(0.001, 0.001),  
    }
    return priors

In [48]:
# ---------------- MCMC run configuration ---------------
warmup = 1000
samples = 2000
chains = 2
# Target acceptance rate: lower values speed up sampling but may increase divergences.
TARGET_ACCEPT = 0.95
CHAIN_METHOD = "parallel"
PROGRESS_BAR = True

In [49]:
def model_hsa_tv_decomp(pi, pi_prev, pi_expect, x, N, l):
    priors = set_prior_distributions()
    
    # NKPC params
    alpha = numpyro.sample("alpha", priors["alpha"])
    theta = numpyro.sample("theta", priors["theta"])
    delta = numpyro.sample("delta", priors["delta"])
    phi_1 = numpyro.sample("phi_1", priors["phi"])
    phi_2 = numpyro.sample("phi_2", priors["phi"])
    
    # State equation (N) params
    r = numpyro.sample("r", dist.Uniform(-1, 1))      
    p = numpyro.sample("p", dist.Uniform(0.1, 0.9))  
    rho_1 = 2 * r * jnp.cos(jnp.pi * p)
    rho_2 = -r ** 2
    numpyro.deterministic("rho_1", rho_1)
    numpyro.deterministic("rho_2", rho_2)
    n = numpyro.sample("n", priors["n"])
    
    # Sigma
    sigma_e = numpyro.sample("sigma_e", priors["sigma_e"])
    sigma_v = numpyro.sample("sigma_v", priors["sigma_v"])      
    sigma_u = numpyro.sample("sigma_u", priors["sigma_u"])      
    sigma_eps = numpyro.sample("sigma_eps", priors["sigma_eps"]) 
    
    # Initial states
    bar_N_0 = numpyro.sample("bar_N_0", dist.Normal(N[0], 1))
    hat_N_0 = numpyro.sample("hat_N_0", dist.Normal(0, 0.05))
    hat_N_1 = numpyro.sample("hat_N_1", dist.Normal(0, 0.05))
    kappa_0 = numpyro.sample("kappa_init", priors["kappa"])
    
    # ===== Handle t=0 OUTSIDE scan =====
    # Trend at t=0
    Nbar_0 = numpyro.sample("Nbar_0", dist.Normal(n + bar_N_0, sigma_eps))
    # Cycle at t=0
    Nhat_obs_0 = N[0] - Nbar_0
    numpyro.sample("Nhat_pred_0", dist.Normal(rho_1 * hat_N_0 + rho_2 * hat_N_1, sigma_u), obs=Nhat_obs_0)
    Nhat_0 = numpyro.deterministic("Nhat_0", Nhat_obs_0)
    # Kappa at t=0
    kappa_0_updated = kappa_0 + delta * (Nbar_0 - bar_N_0)
    numpyro.deterministic("kappa_0", kappa_0_updated)
    # X at t=0 (no lag, just use initial value or prior mean)
    x_star_0 = numpyro.sample("xstar_0", dist.Normal(0.0, sigma_e), obs=x[0])
    # Pi at t=0
    pi_pred_0 = alpha * pi_prev[0] + (1 - alpha) * pi_expect[0] + kappa_0_updated * x_star_0 - theta * Nhat_0
    numpyro.sample("pi_obs_0", dist.Normal(pi_pred_0, sigma_v), obs=pi[0])
    
    # ===== scan from t=1 onwards =====
    def transition(carry, t):
        Nbar_prev = carry[0]
        Nhat_prev_1 = carry[1]
        Nhat_prev_2 = carry[2]
        kappa_prev = carry[3]
        
        # Trend
        Nbar_t = numpyro.sample(f"Nbar_{t}", dist.Normal(n + Nbar_prev, sigma_eps))
        
        # Cycle
        Nhat_obs = N[t] - Nbar_t
        numpyro.sample(f"Nhat_pred_{t}", dist.Normal(rho_1 * Nhat_prev_1 + rho_2 * Nhat_prev_2, sigma_u), obs=Nhat_obs)
        Nhat_t = numpyro.deterministic(f"Nhat_{t}", Nhat_obs)
        
        # Kappa
        kappa_t = kappa_prev + delta * (Nbar_t - Nbar_prev)
        numpyro.deterministic(f"kappa_{t}", kappa_t)
        
        # X observation equation - NOW WITH obs=x[t]
        x_star = numpyro.sample(f"xstar_{t}", 
                                dist.Normal(phi_1 * x[t-1] + phi_2 * Nhat_t, sigma_e), 
                                obs=x[t])
        
        # Pi observation equation
        pi_pred = alpha * pi_prev[t] + (1 - alpha) * pi_expect[t] + kappa_t * x_star - theta * Nhat_t
        numpyro.sample(f"pi_obs_{t}", dist.Normal(pi_pred, sigma_v), obs=pi[t])
        
        return [Nbar_t, Nhat_t, Nhat_prev_1, kappa_t], None
    
    # Start from t=1
    timesteps = jnp.arange(1, l)
    init_carry = [Nbar_0, Nhat_0, hat_N_0, kappa_0_updated]
    
    scan(transition, init_carry, timesteps)

# ==================================================================================================
# for reproducibility
rng_key = jax.random.PRNGKey(0)
rng_keys = jax.random.split(rng_key, chains)
numpyro.enable_x64()

# Define all x-variables you want to test
model_dict = {
    "HSA_tv_decomp_empgap_1": model_hsa_tv_decomp,
}
# 1) model
dict_idata = {}
print("=== Run models ===")
for model_name, m in model_dict.items():
    print(f"Running NKPC model: {model_name}")
    kernel = NUTS(m, target_accept_prob=TARGET_ACCEPT)
    mcmc = MCMC(kernel, num_warmup=warmup, num_samples=samples, num_chains=chains, chain_method=CHAIN_METHOD, progress_bar=PROGRESS_BAR)
    mcmc.run(rng_keys, pi=pi, pi_prev=pi_prev, pi_expect=pi_expect, x=x_unempgap, N=N, l=len(pi))
    idata = az.from_numpyro(mcmc)
    dict_idata[model_name] = idata
    # Divergence diagnostics
    diverging = idata.sample_stats["diverging"].values
    ratio_div = float(np.mean(diverging))
    print(f"Divergence ratio for {model_name}: {ratio_div:.4%}")
print("\n=== All models finished ===\n")

=== Run models ===
Running NKPC model: HSA_tv_decomp_empgap_1


sample: 100%|██████████| 3000/3000 [02:43<00:00, 18.34it/s, 255 steps of size 2.92e-02. acc. prob=0.94] 


Divergence ratio for HSA_tv_decomp_empgap_1: 0.0000%

=== All models finished ===



In [ ]:
# for results
# SDDR
def sddr_alpha(idata):
    """BF_01 = posterior_density_at_0 / prior_density_at_0"""
    # posterior draws of kappa
    post = np.asarray(idata.posterior["alpha"]).ravel()
    post = post[np.isfinite(post)]
    if post.size < 10:
        return np.nan  # safety

    # posterior density at 0 (KDE)
    kde = gaussian_kde(post)
    post_at0 = float(kde.evaluate([0.0])[0])
    # prior density at 0
    prior_at0 = norm.pdf(0.0, loc=alpha_mu, scale=alpha_sigma)
    return post_at0 / max(prior_at0, 1e-300)

def sddr_kappa(idata):
    """BF_01 = posterior_density_at_0 / prior_density_at_0"""
    # posterior draws of kappa
    post = np.asarray(idata.posterior["kappa"]).ravel()
    post = post[np.isfinite(post)]
    if post.size < 10:
        return np.nan  # safety

    # posterior density at 0 (KDE)
    kde = gaussian_kde(post)
    post_at0 = float(kde.evaluate([0.0])[0])
    # prior density at 0
    prior_at0 = norm.pdf(0.0, loc=kappa_mu, scale=kappa_sigma)
    return post_at0 / max(prior_at0, 1e-300)

def sddr_theta(idata):
    """BF_01 = posterior_density_at_0 / prior_density_at_0"""
    # posterior draws of kappa
    post = np.asarray(idata.posterior["theta"]).ravel()
    post = post[np.isfinite(post)]
    if post.size < 10:
        return np.nan  # safety
    # posterior density at 0 (KDE)
    kde = gaussian_kde(post)
    post_at0 = float(kde.evaluate([0.0])[0])
    # prior density at 0
    prior_at0 = norm.pdf(0.0, loc=theta_mu, scale=theta_sigma)
    return post_at0 / max(prior_at0, 1e-300)

def sddr_delta(idata):
    """BF_01 = posterior_density_at_0 / prior_density_at_0"""
    # posterior draws of kappa
    post = np.asarray(idata.posterior["delta"]).ravel()
    post = post[np.isfinite(post)]
    if post.size < 10:
        return np.nan  # safety

    # posterior density at 0 (KDE)
    kde = gaussian_kde(post)
    post_at0 = float(kde.evaluate([0.0])[0])
    # prior density at 0
    prior_at0 = norm.pdf(0.0, loc=delta_mu, scale=delta_sigma)
    return post_at0 / max(prior_at0, 1e-300)

    
# for plot
def plot_prior_posterior_grid(idatas, labels, params=("kappa","alpha","theta"),
                              figsize=(9,2.8), xlims=None):
    assert len(idatas) == len(labels)
    n_rows, n_cols = len(idatas), len(params)
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(figsize[0]*n_cols, figsize[1]*n_rows),
                             squeeze=False, sharey='col')

    priors = {"alpha": (alpha_mu, alpha_sigma),
              "kappa": (kappa_mu, kappa_sigma),
              "theta": (theta_mu, theta_sigma),
              "delta": (delta_mu, delta_sigma)}

    for i, (idata, label_row) in enumerate(zip(idatas, labels)):
        avail = set(idata.posterior.data_vars) if "posterior" in idata.__dict__ else set()
        for j, param in enumerate(params):
            ax = axes[i, j]
            if param in avail:
                az.plot_posterior(idata, var_names=[param], point_estimate=None,
                                  hdi_prob="hide", kind="kde", color="royalblue", ax=ax)
                if param in priors:
                    mu, sigma = priors[param]
                    if xlims and param in xlims:
                        xmin, xmax = xlims[param]
                    else:
                        xmin, xmax = mu - 5*sigma, mu + 5*sigma
                    x = np.linspace(xmin, xmax, 1000)
                    y = norm.pdf(x, mu, sigma)
                    ax.plot(x, y, "r--", lw=2, label="Prior")
            else:
                ax.set_xticks([]); ax.set_yticks([])

            if j == 0:
                ax.set_ylabel("Density", fontsize=11)
                # ax.set_title(label_row, fontsize=20, loc="left")
            ax.set_xlabel({"kappa": r"$\kappa$", "alpha": r"$\alpha$",
                           "theta": r"$\theta$", "delta": r"$\delta$"}.get(param, param), fontsize=16)
            if xlims and param in xlims: ax.set_xlim(xlims[param])

    legend_handles = [Line2D([0],[0], color="royalblue", lw=2, label="Posterior"),
                      Line2D([0],[0], color="red", lw=2, ls="--", label="Prior")]
    fig.legend(handles=legend_handles, fontsize=10, loc="lower center",
               ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.02))
    plt.tight_layout()
    return fig

In [ ]:
# results
models_to_show ={
    "HSA_tv_decomp",
}
# dict_idata subset
dict_items_fill = {k: v for k, v in dict_idata.items() if k in models_to_show}
show_hdi = True
target_params = ["alpha", "theta", "delta","rho_1", "rho_2", "n"]
# ------- Summary table -------
summ_list = []
for model_name, idata in dict_items_fill.items():
    present = set(idata.posterior.data_vars)
    selected = list(present.intersection(target_params))
    if len(selected) == 0:
        continue
    # ArviZ summary
    s = az.summary(idata, var_names=selected, hdi_prob=0.95)
    s["param"] = s.index
    s["model"] = model_name
    s = s.reset_index(drop=True)
    s = s[["model", "param", "mean", "hdi_2.5%", "hdi_97.5%"]]
    summ_list.append(s)
summary_long = pd.concat(summ_list, ignore_index=True)

# ---- HDI on/off ----
if show_hdi:
    summary_long["value"] = summary_long.apply(
        lambda r: f"{r['mean']:.4f} [{r['hdi_2.5%']:.4f}, {r['hdi_97.5%']:.4f}]",
        axis=1
    )
else:
    summary_long["value"] = summary_long["mean"].map(lambda x: f"{x:.4f}")

# pivot
summary_wide = (
    summary_long.pivot_table(
        index="model",
        columns="param",
        values="value",
        aggfunc="first",
    )
    .reindex(columns=target_params)
    .reset_index()
    .rename_axis(None, axis=1)
)

display(HTML("<h3>Summary</h3>"))
display(summary_wide.style.hide(axis="index"))

# SDDR（kappa / theta）table
sddr_rows = []
for model_name, idata in dict_items_fill.items():
    try:
        bf01_alpha = sddr_alpha(idata)
    except Exception:
        bf01_alpha = np.nan
    try:
        bf01_kappa = sddr_kappa(idata)
    except Exception:
        bf01_kappa = np.nan
    try:
        bf01_theta = sddr_theta(idata)
    except Exception:
        bf01_theta = np.nan
    try:
        bf01_delta = sddr_delta(idata)
    except Exception:
        bf01_delta = np.nan
    sddr_rows.append({
        "model": model_name,
        "SDDR_BF01_alpha": bf01_alpha,
        "SDDR_BF01_kappa": bf01_kappa,
        "SDDR_BF01_theta": bf01_theta,
        "SDDR_BF01_delta": bf01_delta,
    })

df_sddr = pd.DataFrame(sddr_rows)
df_sddr["SDDR_BF01_alpha"] = df_sddr["SDDR_BF01_alpha"].map(lambda v: f"{v:.4}")
df_sddr["SDDR_BF01_theta"] = df_sddr["SDDR_BF01_theta"].map(lambda v: f"{v:.4}")
df_sddr["SDDR_BF01_delta"] = df_sddr["SDDR_BF01_delta"].map(lambda v: f"{v:.4}")


display(HTML("<h3>SDDR: kappa: Bayes Factor (BF01)</h3>"))
display(df_sddr.style.hide(axis="index"))

display(HTML("<h3>prior vs posterior </h3>"))
# 6) prior vs posterior 
params = ("alpha", "theta", "delta")
try:
    idatas = [idata for _, idata in dict_items_fill.items()]
    labels  = [name for name, _ in dict_items_fill.items()]
    fig = plot_prior_posterior_grid(
            figsize=(4,2.5),
            idatas=idatas,
            labels=labels,
            params=params,
            xlims={"kappa": (-0.15, 0.25), "alpha": (0, 1.0), "theta": (-0.4, 0.6), "delta": (-0.4, 0.6)})
    plt.show()
except Exception as e:
    print(f"[plot_prior_posterior_grid] error: {e}")


display(HTML("<h3>Decomposed N for Each Model</h3>"))

for model_name, idata in dict_items_fill.items():
    if "Nbar_t" in idata.posterior:

        # ---- Samples ----
        Nhat_samples = idata.posterior["Nhat_t"].values
        Nhat_samples = Nhat_samples.reshape(-1, Nhat_samples.shape[-1])
        Nhat_mean = np.mean(Nhat_samples, axis=0)

        Nbar_samples = idata.posterior["Nbar_t"].values
        Nbar_samples = Nbar_samples.reshape(-1, Nbar_samples.shape[-1])
        Nbar_mean = np.mean(Nbar_samples, axis=0)

        # ---- Figure: two panels ----
        fig, (ax1, ax2) = plt.subplots(
            ncols=2, figsize=(14, 4), sharex=True
        )

        # =========================
        # Left panel: Level
        # =========================
        ax1.plot(data["DATE"], data["N"], label="N (obs)", color="black")
        ax1.plot(data["DATE"], Nbar_mean, label="Nbar (trend)", color="green")
        ax1.set_title("Level", fontsize=14)
        ax1.set_ylabel("N", fontsize=14)
        ax1.legend(loc="upper right", fontsize=14)

        # =========================
        # Right panel: Cycle
        # =========================
        ax2.plot(data["DATE"], Nhat_mean, label="Nhat (cycle)", color="blue")
        ax2.set_title("Cycle", fontsize=14)
        ax2.set_ylabel("Nhat", fontsize=14)
        ax2.legend(loc="upper right", fontsize=14)

        # ---- X-axis formatting (shared) ----
        for ax in (ax1, ax2):
            ax.set_xlabel("Year")
            ax.xaxis.set_major_locator(mdates.YearLocator(5))
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
            ax.xaxis.set_minor_locator(mdates.YearLocator(1))
            ax.minorticks_on()
            ax.grid(which="major", linestyle="-", linewidth=0.75)
            ax.grid(which="minor", linestyle=":", linewidth=0.5)
        plt.tight_layout()
        plt.show()

# ===== Plot kappa_t for each model with 95% HDI =====
display(HTML("<h3>Latent Variable kappa_t (95% HDI) for Each Model</h3>"))
for model_name, idata in dict_items_fill.items():
        # Extract kappa_t samples from posterior
        if "kappa_t" in idata.posterior:
            kappa_samples = idata.posterior["kappa_t"].values
            # Flatten chain and draw dimensions: (chain, draw, time) -> (samples, time)
            kappa_samples = kappa_samples.reshape(-1, kappa_samples.shape[-1])
            
            # Calculate mean and 95% HDI using ArviZ
            kappa_mean = np.mean(kappa_samples, axis=0)
            kappa_hdi = az.hdi(kappa_samples, hdi_prob=0.95)
            kappa_lower = kappa_hdi[:, 0]
            kappa_upper = kappa_hdi[:, 1]
            
            # Moving average
            window = 5
            kappa_ma = pd.Series(kappa_mean).rolling(window=window, center=True).mean()
            
        # Plot
        plt.figure(figsize=(12, 4))

        plt.plot(
            data["DATE"], kappa_mean,
            label=r"Mean of $\kappa_t$",
            color="green",
            linewidth=2.5
        )

        plt.fill_between(
            data["DATE"], kappa_lower, kappa_upper,
            color="green",
            alpha=0.25,
            label="95% HDI"
        )

        # ---- Labels (bigger) ----
        plt.xlabel("Year", fontsize=14)
        plt.ylabel(r"$\kappa_t$", fontsize=14)

        # ---- Legend (bigger) ----
        plt.legend(fontsize=12)

        # ---- Ticks (bigger) ----
        plt.tick_params(axis="both", which="major", labelsize=12)
        plt.tick_params(axis="both", which="minor", labelsize=10)

        # ---- Grid ----
        plt.minorticks_on()
        plt.grid(which="major", linestyle="-", linewidth=0.75)
        plt.grid(which="minor", linestyle=":", linewidth=0.5)

        # ---- X-axis formatting ----
        ax = plt.gca()
        ax.xaxis.set_major_locator(mdates.YearLocator(5))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        ax.xaxis.set_minor_locator(mdates.YearLocator(1))

        plt.tight_layout()
        plt.show()

ValueError: No objects to concatenate